# QK Spectral Segmentation — Pipeline

Runs `segment.py` then `eval.py` from the repo.

In [ ]:
# ── Paths — edit these ────────────────────────────────────────────────────────

# Precomputed eigenvectors (Kaggle dataset: wenhaolu49/qk-spectral-eigs)
EIGS_DIR   = '/kaggle/input/qk-spectral-eigs/multilayer_svd'

# VOC2012 root (Kaggle dataset: Pascal VOC 2012)
VOC_DIR    = '/kaggle/input/pascal-voc-2012/VOCdevkit/VOC2012'

# Where segmentation outputs will be saved
OUTPUT_DIR = '/kaggle/working/output'

# ─────────────────────────────────────────────────────────────────────────────

## 1. Clone repo

In [ ]:
from pathlib import Path

REPO = Path('/kaggle/working/qk-spectral-segmentation')

if not REPO.exists():
    !git clone https://github.com/welu2027/qk-spectral-segmentation {REPO}
else:
    print('Repo already present:', REPO)

## 2. Setup

In [ ]:
import os, sys
from pathlib import Path

REPO = Path('/kaggle/working/qk-spectral-segmentation')
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'extract'))
os.chdir(REPO)
print('Repo root:', REPO)

!pip install -q fire pymatting scikit-image

## 3. Verify paths

In [ ]:
checks = {
    'eigs_dir'         : Path(EIGS_DIR),
    'voc JPEGImages'   : Path(VOC_DIR) / 'JPEGImages',
    'voc Annotations'  : Path(VOC_DIR) / 'Annotations',
    'voc SegmClass'    : Path(VOC_DIR) / 'SegmentationClass',
    'voc SegmObject'   : Path(VOC_DIR) / 'SegmentationObject',
}
for name, p in checks.items():
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}]  {name}: {p}')

## 4. Run segment.py  (~30–50 min CPU)

Produces `output/segmentations/`, `output/segmentations_single/`, `output/bboxes.pth`.

In [ ]:
!python3 {REPO}/segment.py \
    --eigs_dir   {EIGS_DIR} \
    --voc_dir    {VOC_DIR} \
    --output_dir {OUTPUT_DIR}

## 5. Run eval.py  (~10 min CPU)

In [ ]:
# --no_semantic skips the ~30-60 min mIoU step; remove the flag to include it
# --no_vis skips matplotlib popups (headless on Kaggle)
!python3 {REPO}/eval.py \
    --eigs_dir   {EIGS_DIR} \
    --output_dir {OUTPUT_DIR} \
    --voc_dir    {VOC_DIR} \
    --no_semantic \
    --no_vis

## 6. Visualize results (inline)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import cv2
import torch
from PIL import Image as PILImage
from skimage.color import label2rgb
from pathlib import Path

images_root = Path(VOC_DIR) / 'JPEGImages'
segs_dir    = Path(OUTPUT_DIR) / 'segmentations'
seg_files   = sorted(segs_dir.glob('*.png'))[:6]

for seg_file in seg_files:
    image_id = seg_file.stem
    image    = np.array(PILImage.open(images_root / f'{image_id}.jpg').convert('RGB'))
    segmap   = np.array(PILImage.open(seg_file))
    seg_full = cv2.resize(segmap, (image.shape[1], image.shape[0]),
                          interpolation=cv2.INTER_NEAREST)
    labels  = np.unique(seg_full)
    colors  = [plt.cm.tab10.colors[i % 10] for i in labels if i != 0]
    overlay = label2rgb(seg_full, image=image, colors=colors, bg_label=0, alpha=0.45)

    eig_data = torch.load(Path(EIGS_DIR) / f'{image_id}.pth', map_location='cpu')
    eigvecs  = eig_data['eigenvectors']
    H_p, W_p = segmap.shape[:2]
    n_show   = min(4, eigvecs.shape[0] - 1)

    fig, axes = plt.subplots(1, 2 + n_show, figsize=(4 * (2 + n_show), 4))
    axes[0].imshow(image);   axes[0].set_title('Image');        axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title('Segmentation'); axes[1].axis('off')
    for j in range(n_show):
        ev = eigvecs[j + 1].numpy().reshape(H_p, W_p)
        axes[2 + j].imshow(ev, cmap='RdBu_r')
        axes[2 + j].set_title(f'Eigvec {j + 1}')
        axes[2 + j].axis('off')
    plt.suptitle(image_id, fontsize=12)
    plt.tight_layout()
    plt.show()